# SAM3 mask post-processing benchmark

Compares ultralytics' `SAM3SemanticPredictor.postprocess` with streetscapes'
`LowMemorySAM3SemanticPredictor`, which upscales the predicted masks to the
original image size one at a time instead of as one float32 tensor.

Only `postprocess` differs between the two, so this notebook times that step in
isolation: it captures the raw model output once, then runs every variant on the
same input, interleaved round-robin so that thermal throttling or other load
affects all variants equally. For each output size it reports time, peak GPU memory
(CUDA only) and whether the masks are identical to upstream's.

**Requirements**
- streetscapes from the branch containing `streetscapes/models/sam3/predictor.py`,
  installed with the `sam3` extra.
- Either SAM3 weights (`sam3.pt`) plus an image to capture inputs from, or a
  previously captured inputs file (see `CAPTURED_PATH`).

## Parameters

In [ ]:
from pathlib import Path

# Option A: capture the model output from an image (needs SAM3 weights).
IMAGE_PATH = None  # e.g. Path("~/images/street.jpg").expanduser()
WEIGHTS = None  # None: use `sam3_model_path` from the streetscapes config
PROMPT = "tree, car, window, building, person, road, sign"
SAVE_CAPTURE_TO = Path("sam3_postprocess_inputs.pt")  # None: don't save

# Option B: load previously captured inputs instead (skips loading the model).
CAPTURED_PATH = None  # e.g. Path("sam3_postprocess_inputs.pt")

# Output sizes (height, width) to upscale the masks to.
SIZES = [(1512, 2016), (3024, 4032), (4096, 8192)]
ROUNDS = 15  # timed rounds per size, after one warm-up round

## Setup

In [ ]:
import gc
import hashlib
import statistics
import time
import types

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from ultralytics.models.sam import SAM3SemanticPredictor

from streetscapes.models.sam3.predictor import LowMemorySAM3SemanticPredictor

CUDA = torch.cuda.is_available()
print(f"torch {torch.__version__}, CPU threads: {torch.get_num_threads()}")
if CUDA:
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("No CUDA device found: timing on CPU, peak memory not reported.")

## Capture the model output

Runs SAM3 once and records the inputs to `postprocess`. Ultralytics picks the
device itself (the first GPU if available), so the captured tensors live on the
device the model ran on.

In [ ]:
PRED_KEYS = ["pred_boxes", "pred_logits", "pred_masks", "presence_logit_dec"]

if CAPTURED_PATH is not None:
    captured = torch.load(CAPTURED_PATH, weights_only=False)
    device = torch.device("cuda" if CUDA else "cpu")
    captured["preds"] = {k: v.to(device) for k, v in captured["preds"].items()}
else:
    import imageio.v3 as iio

    from streetscapes import CFG
    from streetscapes.models.sam3.model import SAM3

    if IMAGE_PATH is None:
        raise ValueError("Set IMAGE_PATH (option A) or CAPTURED_PATH (option B).")
    weights = WEIGHTS if WEIGHTS is not None else CFG.sam3_model_path
    if weights is None:
        raise ValueError("Set WEIGHTS or `sam3_model_path` in the streetscapes config.")

    model = SAM3(weights=weights)
    predictor = model.model
    captured = {}
    upstream_postprocess = predictor.postprocess

    def capturing_postprocess(preds, img, orig_imgs):
        captured.update(
            preds={k: preds[k].detach().clone() for k in PRED_KEYS},
            orig_shape=orig_imgs[0].shape,
            conf=predictor.args.conf,
            iou=predictor.args.iou,
            agnostic_nms=predictor.args.agnostic_nms,
            mask_threshold=predictor.model.mask_threshold,
            names=predictor.model.names,
        )
        return upstream_postprocess(preds, img, orig_imgs)

    predictor.postprocess = capturing_postprocess
    image = np.asarray(iio.imread(IMAGE_PATH))
    [seg] = model.segment_images(["capture"], [image], PROMPT)
    height, width = image.shape[:2]
    print(f"Captured from a {width}x{height} image: {len(seg['labels'])} instances")

    if SAVE_CAPTURE_TO is not None:
        torch.save(
            captured | {"preds": {k: v.cpu() for k, v in captured["preds"].items()}},
            SAVE_CAPTURE_TO,
        )
        print(f"Saved captured inputs to {SAVE_CAPTURE_TO}")

    # Free the model so it doesn't skew the memory measurements.
    del model, predictor, seg, image
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

print(
    f"pred_masks: {tuple(captured['preds']['pred_masks'].shape)} "
    f"on {captured['preds']['pred_masks'].device}"
)

## Variants

- `upstream`: ultralytics as-is, all masks upscaled in one float32 tensor.
- `lowmem`: what streetscapes ships, one mask at a time.
- `chunk N`: N masks at a time, thresholding straight into the output tensor.
  These were slower or no faster on CPU; included to check whether that holds on
  GPU, where per-call overhead matters more.

In [ ]:
def chunked_predictor(chunk: int) -> type:
    def _upscale_masks(self, masks, size):
        upscaled = torch.empty(
            (masks.shape[0], *size), dtype=torch.bool, device=masks.device
        )
        for start in range(0, masks.shape[0], chunk):
            torch.gt(
                F.interpolate(
                    masks[start : start + chunk].float()[None], size, mode="bilinear"
                )[0],
                self.model.mask_threshold,
                out=upscaled[start : start + chunk],
            )
        return upscaled

    return type(
        f"Chunk{chunk}SAM3SemanticPredictor",
        (LowMemorySAM3SemanticPredictor,),
        {"_upscale_masks": _upscale_masks},
    )


VARIANTS = {
    "upstream": SAM3SemanticPredictor,
    "lowmem": LowMemorySAM3SemanticPredictor,
    "chunk 4": chunked_predictor(4),
    "chunk 8": chunked_predictor(8),
    "chunk 16": chunked_predictor(16),
}


def make_predictor(cls: type):
    # Bypass __init__: postprocess needs only these attributes, not the weights.
    predictor = object.__new__(cls)
    predictor.args = types.SimpleNamespace(
        conf=captured["conf"],
        iou=captured["iou"],
        agnostic_nms=captured["agnostic_nms"],
    )
    predictor.model = types.SimpleNamespace(
        mask_threshold=captured["mask_threshold"], names=captured["names"]
    )
    predictor.batch = (["benchmark"],)
    return predictor

## Benchmark

In [ ]:
def sync():
    if CUDA:
        torch.cuda.synchronize()


def is_oom(err: Exception) -> bool:
    return isinstance(err, torch.OutOfMemoryError) or "memory" in str(err).lower()


def run_once(predictor, size):
    """Run postprocess once; return (seconds, peak MB or None, masks, n_masks)."""
    preds = {k: v.clone() for k, v in captured["preds"].items()}
    orig_imgs = [np.zeros((*size, 3), dtype=np.uint8)]
    sync()
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
        base = torch.cuda.memory_allocated()
    start = time.perf_counter()
    [result] = predictor.postprocess(preds, None, orig_imgs)
    sync()
    seconds = time.perf_counter() - start
    peak_mb = (torch.cuda.max_memory_allocated() - base) / 2**20 if CUDA else None
    masks = result.masks.data if result.masks is not None else None
    n_masks = 0 if masks is None else masks.shape[0]
    return seconds, peak_mb, masks, n_masks


def benchmark(size, rounds):
    predictors = {name: make_predictor(cls) for name, cls in VARIANTS.items()}
    times = {name: [] for name in VARIANTS}
    peaks, hashes, failed = {}, {}, {}
    n_masks = None
    with torch.inference_mode():
        for rnd in range(rounds + 1):  # round 0 is the warm-up
            order = list(VARIANTS) if rnd % 2 else list(reversed(VARIANTS))
            for name in order:
                if name in failed:
                    continue
                try:
                    seconds, peak_mb, masks, n_masks = run_once(predictors[name], size)
                except (torch.OutOfMemoryError, RuntimeError) as err:
                    if not is_oom(err):
                        raise
                    failed[name] = "OOM"
                    continue
                finally:
                    gc.collect()
                    if CUDA:
                        torch.cuda.empty_cache()
                if rnd == 0:
                    packed = b"" if masks is None else np.packbits(masks.cpu().numpy())
                    hashes[name] = hashlib.sha256(bytes(packed)).hexdigest()
                    peaks[name] = peak_mb
                else:
                    times[name].append(seconds)
                del masks

    rows = []
    upstream_times = times["upstream"]
    upstream_median = statistics.median(upstream_times) if upstream_times else None
    for name in VARIANTS:
        row = {"size": f"{size[1]}x{size[0]}", "masks": n_masks, "variant": name}
        if name in failed:
            rows.append(row | {"status": failed[name]})
            continue
        median = statistics.median(times[name])
        rows.append(
            row
            | {
                "status": "ok",
                "median ms": median * 1000,
                "p25 ms": np.percentile(times[name], 25) * 1000,
                "p75 ms": np.percentile(times[name], 75) * 1000,
                "vs upstream": median / upstream_median if upstream_median else None,
                "peak MB": peaks[name],
                "same masks as upstream": (
                    hashes[name] == hashes["upstream"] if "upstream" in hashes else None
                ),
            }
        )
    return rows

In [ ]:
rows = []
for size in SIZES:
    print(f"Benchmarking {size[1]}x{size[0]}...")
    rows.extend(benchmark(size, ROUNDS))

results = pd.DataFrame(rows).set_index(["size", "variant"])
results.style.format(
    {
        "median ms": "{:,.0f}",
        "p25 ms": "{:,.0f}",
        "p75 ms": "{:,.0f}",
        "vs upstream": "{:.2f}x",
        "peak MB": "{:,.0f}",
    },
    na_rep="",
)

## Reading the results

- **vs upstream** below 1 means faster than ultralytics' own `postprocess`.
- **peak MB** is GPU memory allocated during `postprocess`, including the boolean
  output masks (1 byte per pixel per instance), which no variant can avoid.
- **OOM** means the variant ran out of memory at that size and was skipped for the
  remaining rounds.
- The p25–p75 spread shows how noisy the machine is; differences within it are not
  meaningful.